## Overview

This notebook loads the pickled results produced by `scaling_effdim_sparsV.py`, which scans the sparsity index s of the block-sparse random orthogonal matrix V (parameter-side singular-vector matrix in Gamma = U @ diag(S) @ V^T, interpolating between a dense random orthogonal matrix at s=1 and a pure permutation matrix at s=dim_basis_params), and plots the normalized effective dimension against the fraction of nonzero entries of V (NNZ/K^2) — first for a single (M, Dloc) configuration, then overlaying several configurations.

In [ ]:
# Importing necessary packages
import sys
import os
import importlib
import math
import pickle
import numpy

import pennylane.numpy as np




import matplotlib.pyplot as plt


# Current path for importing custom functions
path_base = '/home/b/b309245/FIM_Training_Bias_RegressionModels/fourier_models_training_and_fim/'
sys.path.insert(0, path_base + 'useful_functions')

import model_constructor_functions
importlib.reload(model_constructor_functions)

import ortho_matrices_functions
importlib.reload(ortho_matrices_functions)

import tensor_network_functions_np
importlib.reload(tensor_network_functions_np)

import FIM_functions_jax
importlib.reload(FIM_functions_jax)

import training_functions_jax
importlib.reload(training_functions_jax)

Sets the fixed scan parameters (input dimension D, decay exponent of the correlation spectrum S, number of parameter/V samples) for a single (no_params, local_dim_param) configuration and builds the filename pattern used to locate its pickled sparsity-index scan (produced by `scaling_effdim_sparsV.py`).

In [ ]:
results_folder = '/work/bd1179/b309245/fourier_models_train_and_FIM/scaling_effdim_sparsity_Vh/'

# Dimension of input function space
dim_in = 20
name_dim_in = str(dim_in)

# Decay exponent
dec_exp = 0.0
dec_exp_name = '0p0'

# No. of random parameter samples for evaluating normalized eff. dim.
no_samples = 150
no_par_samples_name = str(no_samples)

# No. of random orthogonal matrix realizations per S decay exponent
no_matrix_realiz = 40
no_V_samples_name = str(no_matrix_realiz)

# No. of parameters
#no_params = 7
no_params = 10
name_no_params = str(no_params)

# Local dimension of parameter functions space
#local_dim_param = 3
local_dim_param = 2
name_dim_par_loc = str(local_dim_param)

dim_basis_params = local_dim_param ** no_params

name_end_0 = ('_DimIn' + name_dim_in + '_DimLocPar' + name_dim_par_loc + '_Nparams' + name_no_params + 
              '_NsamplesPar' + no_par_samples_name + '_NsamplesV' + no_V_samples_name + 
              '_DecExpS' + dec_exp_name + '_SparsityInd')

Defines helper functions `no_of_zeros`/`no_non_zeros` that compute the number of zero/nonzero entries of the D x D block-sparse orthogonal matrix V for a given sparsity index s, used below to convert the swept sparsity index into a nonzero-entry fraction NNZ/K^2 for plotting.

In [ ]:
def no_of_zeros(s, D):
    if (math.remainder(D, s)<1.0e-10):
        Db = int(D / s)
        nnzs = s * Db * Db
        nzs = D * D - nnzs
    else:
        Db1 = int(np.floor(D / s))
        Db2 = D - (s-1) * Db1
        nnzs = (s - 1) * Db1 * Db1 + Db2 * Db2
        nzs = D * D - nnzs
    return nzs


def no_non_zeros(s, D):
    return D*D - no_of_zeros(s, D)

Loads and concatenates the pickled sparsity index, correlation-spectrum purity tr(S^4), and normalized effective dimension arrays across all matching result files for this single configuration, converting sparsity indices to zero/nonzero entry-count ratios via the helper functions above.

In [ ]:
namefileini = 'norm_eff_dim' + name_end_0
listallfiles = [f for f in os.listdir(results_folder) if (f.startswith(namefileini))]
print(len(listallfiles))

n0ratio_all = []
nnz_ratio_all = []
sparsityV_all = []
purity_S_all = []
norm_eff_dim_all = []

for filename in listallfiles:
    path_file = os.path.join(results_folder, filename)
    with open(path_file, 'rb') as f:
        dict_norm_ed = pickle.load(f)

    sparse_vec = dict_norm_ed['sparsityV_all']
    purity_S_vec = dict_norm_ed['purity_S_all']
    norm_eff_dim_vec = dict_norm_ed['norm_eff_dim_all']
    sp = sparse_vec[0]
    n0 = no_of_zeros(sp, dim_basis_params)
    n0sr = (n0 / dim_basis_params**2.0) * np.ones(len(sparse_vec))
    nnz = no_non_zeros(sp, dim_basis_params)
    nnz_r = (nnz / dim_basis_params**2.0) * np.ones(len(sparse_vec))

    n0ratio_all.append(n0sr)
    nnz_ratio_all.append(nnz_r)
    sparsityV_all.append(sparse_vec)
    purity_S_all.append(purity_S_vec)
    norm_eff_dim_all.append(norm_eff_dim_vec)

n0ratio_all = np.concatenate(n0ratio_all)
nnz_ratio_all = np.concatenate(nnz_ratio_all)
sparsityV_all = np.concatenate(sparsityV_all)
purity_S_all = np.concatenate(purity_S_all)
norm_eff_dim_all = np.concatenate(norm_eff_dim_all)

Plots the normalized effective dimension d_eff against the nonzero-entry fraction NNZ/K^2 of V (log-scaled x-axis) for this single configuration.

In [ ]:
fs = 24
figsize = (10,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

#x = np.asarray(n0ratio_all)
x = np.asarray(nnz_ratio_all)
y = np.asarray(norm_eff_dim_all)
ax.plot(x, y, '.', color='tab:blue', markersize=10)

#ax.legend(fontsize=22)
#ax.set_xlabel(r'$(\mathrm{no.\,}0)/K^2$', fontsize=fs)
ax.set_xlabel(r'$(\mathrm{NNZ})/K^2$', fontsize=fs)
ax.set_ylabel(r'$\hat{d}_{\mathrm{eff}}$', fontsize=fs)
ax.set_xscale('log')
#ax.set_yscale('log')

Re-defines the fixed scan parameters and lists the (M, Dloc) combinations whose pickled sparsity-index scans will be loaded and overlaid in the plot below (second, multi-configuration pass).

In [ ]:
results_folder = '/work/bd1179/b309245/fourier_models_train_and_FIM/scaling_effdim_sparsity_Vh/'

# Dimension of input function space
dim_in = 20
name_dim_in = str(dim_in)

# Decay exponent
dec_exp = 0.0
dec_exp_name = '0p0'

# No. of random parameter samples for evaluating normalized eff. dim.
no_samples = 150
no_par_samples_name = str(no_samples)

# No. of random orthogonal matrix realizations per S decay exponent
no_matrix_realiz = 40
no_V_samples_name = str(no_matrix_realiz)

########################### Exps. to plot ###########################

# No. of parameters
no_params_vec = [10, 7]
name_no_params_vec = [str(no_params) for no_params in no_params_vec]

# Local dimension of parameter functions space
local_dim_param_vec = [2, 3]
name_dim_par_loc_vec = [str(local_dim_param) for local_dim_param in local_dim_param_vec]

no_exps = len(no_params_vec)

For each (M, Dloc) combination, loads and concatenates the pickled sparsity index, correlation-spectrum purity tr(S^4), and normalized effective dimension arrays (with sparsity indices converted to nonzero-entry ratios) across all matching result files into `dict_exps`.

In [ ]:
dict_exps = dict()

for i in range(no_exps):
    no_params = no_params_vec[i]
    local_dim_param = local_dim_param_vec[i]
    name_no_params = name_no_params_vec[i]
    name_dim_par_loc = name_dim_par_loc_vec[i]

    dim_basis_params = local_dim_param ** no_params
    name_end_0 = ('_DimIn' + name_dim_in + '_DimLocPar' + name_dim_par_loc + '_Nparams' + name_no_params + 
                  '_NsamplesPar' + no_par_samples_name + '_NsamplesV' + no_V_samples_name + 
                  '_DecExpS' + dec_exp_name + '_SparsityInd')
    
    namefileini = 'norm_eff_dim' + name_end_0
    listallfiles = [f for f in os.listdir(results_folder) if (f.startswith(namefileini))]
    print(len(listallfiles))
    
    n0ratio_all = []
    nnz_ratio_all = []
    sparsityV_all = []
    purity_S_all = []
    norm_eff_dim_all = []
    
    for filename in listallfiles:
        path_file = os.path.join(results_folder, filename)
        with open(path_file, 'rb') as f:
            dict_norm_ed = pickle.load(f)
    
        sparse_vec = dict_norm_ed['sparsityV_all']
        purity_S_vec = dict_norm_ed['purity_S_all']
        norm_eff_dim_vec = dict_norm_ed['norm_eff_dim_all']
        sp = sparse_vec[0]
        n0 = no_of_zeros(sp, dim_basis_params)
        n0sr = (n0 / dim_basis_params**2.0) * np.ones(len(sparse_vec))
        nnz = no_non_zeros(sp, dim_basis_params)
        nnz_r = (nnz / dim_basis_params**2.0) * np.ones(len(sparse_vec))
    
        n0ratio_all.append(n0sr)
        nnz_ratio_all.append(nnz_r)
        sparsityV_all.append(sparse_vec)
        purity_S_all.append(purity_S_vec)
        norm_eff_dim_all.append(norm_eff_dim_vec)
    
    n0ratio_all = np.concatenate(n0ratio_all)
    nnz_ratio_all = np.concatenate(nnz_ratio_all)
    sparsityV_all = np.concatenate(sparsityV_all)
    purity_S_all = np.concatenate(purity_S_all)
    norm_eff_dim_all = np.concatenate(norm_eff_dim_all)

    dict_exp = dict()
    dict_exp['no_params'] = no_params
    dict_exp['local_dim_param'] = local_dim_param
    dict_exp['nnz_ratio_all'] = nnz_ratio_all
    dict_exp['purity_S_all'] = purity_S_all
    dict_exp['sparsityV_all'] = sparsityV_all
    dict_exp['norm_eff_dim_all'] = norm_eff_dim_all
    dict_exps[i] = dict_exp

Plots the normalized effective dimension d_eff against the nonzero-entry fraction NNZ/K^2 of V (log-scaled x-axis) for each (M, Dloc) configuration, labeling each curve with Dloc (d_tilde), M, and the correlation-spectrum purity tr(S^4).

In [ ]:
fs = 28
figsize = (8,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

clrs = ['tab:blue', 'tab:orange']

for i in range(no_exps):
    dict_exp = dict_exps[i]
    nnz_ratio_all = dict_exp['nnz_ratio_all']
    norm_eff_dim_all = dict_exp['norm_eff_dim_all']
    no_params = dict_exp['no_params']
    local_dim_param = dict_exp['local_dim_param']
    purity_S_all = numpy.asarray(dict_exp['purity_S_all'])
    purity_rounded = round(numpy.mean(purity_S_all), 2)
    print('S purity: ', purity_rounded, ' with std. ', np.std(purity_S_all))
    lbl = r'$\tilde{d}=' + str(local_dim_param) + ',\,M=' + str(no_params) + ',\,\mathrm{tr}\,S^4=' + str(purity_rounded) + '$'
    x = np.asarray(nnz_ratio_all)
    y = np.asarray(norm_eff_dim_all)
    ax.plot(x, y, '.', color=clrs[i], markersize=12, label=lbl)

ax.legend(fontsize=22)
ax.set_xlabel(r'$\mathrm{NNZ}/K^2$', fontsize=fs)
ax.set_ylabel(r'$\hat{d}_{\mathrm{eff}}$', fontsize=fs)
ax.set_xscale('log')
#ax.set_yscale('log')

fig.savefig('EffDimScaling_vs_SparsityV.png', bbox_inches='tight')
fig.savefig('EffDimScaling_vs_SparsityV.pdf', bbox_inches='tight')